# 01 · Annotate each gene's chromosome

Query MyGene for the chromosome of every gene and store it in `var['chromosome']`.
This annotation is used in notebook 02 to drop sex-linked (X/Y) genes before batch correction.

Input: `snRNAseq_LCNE.h5ad` (exported from R). Output: `snRNAseq_LCNE_with_chrom.h5ad`.

In [ ]:
import sys
import re
import scanpy as sc
import mygene
import scvi

sys.path.append('/root/capsule/code/')
from utils import get_paths

print("Last run with scvi-tools version:", scvi.__version__)

## Helper: look up each gene's chromosome (MyGene)

In [ ]:
mg = mygene.MyGeneInfo()


def fetch_chr_mygene(symbols, chunk=1000):
    allowed = {str(i) for i in range(1,20)} | {"X","Y","MT"}
    out = {}
    for i in range(0, len(symbols), chunk):
        chunk_syms = symbols[i:i+chunk]
        hits = mg.querymany(
            chunk_syms,
            scopes="symbol",
            fields="genomic_pos,genomic_pos_hg19,map_location,symbol",
            species="mouse",
            as_dataframe=False,
            returnall=False,
            verbose=False)
        for h in hits:
            q = h.get("query")
            c = None
            # prefer genomic_pos.chr
            gp = h.get("genomic_pos")
            if isinstance(gp, list) and gp:
                gp = gp[0]
            if isinstance(gp, dict):
                c = gp.get("chr")
            if c is None:
                ml = h.get("map_location")
                if isinstance(ml, str):
                    c = re.split(r"[ ;,]", ml.strip())[0]
            if isinstance(c, str):
                c = c.replace("chr","").upper()
            if c in allowed and q not in out:
                out[q] = c
    return out

## Load snRNA data (exported from R)

In [ ]:
# Path resolved via utils.get_paths() (reads config.toml). To point at a different
# data asset, update the prefix in config.toml — no change needed here.
paths = get_paths()
adata_sc_orig = sc.read_h5ad(paths['snRNAseq_h5ad'])
adata_sc_orig.obs['actualsex'] = adata_sc_orig.obs['sex'].str[0]  # take only first character 'M' or 'F'

# get gene symbols
symbols = adata_sc_orig.var_names.astype(str).tolist()


## Annotate `var['chromosome']` and save

In [ ]:
gene_to_chr = fetch_chr_mygene(symbols)  # get the chromosomal info!
adata_sc_orig.var["chromosome"] = adata_sc_orig.var_names.map(gene_to_chr).astype("category")


# lets just save this so that we dont need to run query all the time 
adata_sc_orig.write("/scratch/snRNAseq_LCNE_with_chrom.h5ad")

print('saved!', adata_sc_orig.shape)